<a href="https://colab.research.google.com/github/Striver29/CrossEnrich/blob/main/CrossEnrich_v0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Phase 1: Hardcoded data collection**

In [2]:
!pip install gprofiler-official --quiet
print("done")

done


In [5]:
from gprofiler import GProfiler
import pandas as pd

In [6]:
gene_list = ["TP53", "MDM2", "CDKN1A", "BAX", "BBC3", "PMAIP1", "GADD45A", "GADD45B",
"SESN1", "SESN2", "RRM2B", "DRAM1", "FAS", "TNFRSF10B", "APAF1",
"CASP3", "CASP9", "CYCS", "PTEN", "ATM", "ATR", "BRCA1", "BRCA2"]

print(len(gene_list))
print(gene_list)

23
['TP53', 'MDM2', 'CDKN1A', 'BAX', 'BBC3', 'PMAIP1', 'GADD45A', 'GADD45B', 'SESN1', 'SESN2', 'RRM2B', 'DRAM1', 'FAS', 'TNFRSF10B', 'APAF1', 'CASP3', 'CASP9', 'CYCS', 'PTEN', 'ATM', 'ATR', 'BRCA1', 'BRCA2']


In [22]:
gp = GProfiler()
results = gp.profile(gene_list, organism="hsapiens")
results = pd.DataFrame(results)
print(results.shape)

(608, 14)


In [15]:
results.head()

,description,effective_domain_size,intersection_size,name,native,p_value,parents,precision,query,query_size,recall,significant,source,term_size
0,p53 signaling pathway,8484,20,p53 signaling pathway,KEGG:04115,5.941134e-39,[KEGG:00000],0.909091,query_1,22,0.270270,True,KEGG,74
1,DNA damage response,8752,19,DNA damage response,WP:WP707,8.715110e-36,[WP:000000],0.826087,query_1,23,0.275362,True,WP,69
2,miRNA regulation of DNA damage response,8752,19,miRNA regulation of DNA damage response,WP:WP1530,2.994203e-35,[WP:000000],0.826087,query_1,23,0.260274,True,WP,73
3,Platinum drug resistance,8484,13,Platinum drug resistance,KEGG:01524,1.175199e-20,[KEGG:00000],0.590909,query_1,22,0.180556,True,KEGG,72
4,miRNA regulation of p53 pathway in prostate ca...,8752,11,miRNA regulation of p53 pathway in prostate ca...,WP:WP3982,2.706305e-20,[WP:000000],0.478261,query_1,23,0.354839,True,WP,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,CASP8 activity is inhibited,11004,2,CASP8 activity is inhibited,REAC:R-HSA-5218900,4.989271e-02,[REAC:R-HSA-5675482],0.095238,query_1,21,0.200000,True,REAC,10
604,Caspase activation via Dependence Receptors in...,11004,2,Caspase activation via Dependence Receptors in...,REAC:R-HSA-418889,4.989271e-02,[REAC:R-HSA-5357769],0.095238,query_1,21,0.200000,True,REAC,10
605,Dimerization of procaspase-8,11004,2,Dimerization of procaspase-8,REAC:R-HSA-69416,4.989271e-02,[REAC:R-HSA-140534],0.095238,query_1,21,0.200000,True,REAC,10
606,"""Combining with an extracellular messenger (ca...",20246,2,death receptor activity,GO:0005035,4.990253e-02,[GO:0004888],0.086957,query_1,23,0.125000,True,GO:MF,16


In [19]:
print(results["source"].unique())

['KEGG' 'WP' 'REAC' 'GO:BP' 'HP' 'GO:CC' 'MIRNA' 'CORUM' 'GO:MF' 'TF'
 'HPA']


**Note: We treat GO as three separate databases:**

GO:BP (Biological Processes)

GO:CC (Cellular Component)

GO:MF (Molecular Function)

This leaves us with 5 databases for our analysis (10 pairwise comparison): GO:BP, GO:CC, GO:MF, KEGG and REACTOM.

#**Phase 2: Source filteration**




In [35]:
df_gobp = results[results["source"] == "GO:BP"]
df_gocc = results[results["source"] == "GO:CC"]
df_gomf = results[results["source"] == "GO:MF"]
df_kegg = results[results["source"] == "KEGG"]
df_reac = results[results["source"] == "REAC"]

The code below is a safe guard filter for allowing terms that multiple correction

In [36]:
df_gobp = df_gobp[df_gobp["significant"] == True]
df_gocc = df_gocc[df_gocc["significant"] == True]
df_gomf = df_gomf[df_gomf["significant"] == True]
df_kegg = df_kegg[df_kegg["significant"] == True]
df_reac = df_reac[df_reac["significant"] == True]

In [38]:
print("GO:BP ",df_gobp.shape)
print("GO:CC ",df_gocc.shape)
print("GO:MF ", df_gomf.shape)
print("KEGG ", df_kegg.shape)
print("REAC ", df_reac.shape)

GO:BP  (236, 14)
GO:CC  (12, 14)
GO:MF  (13, 14)
KEGG  (54, 14)
REAC  (76, 14)


#**Phase 3: Consistency and Metrics**




In [60]:
#mock example of two databases and direct name matching.

df_kegg_set = set(df_kegg["name"])
df_reac_set = set(df_reac["name"])

union = df_kegg_set.union(df_reac_set)
intersection = df_kegg_set.intersection(df_reac_set)

jaccard = len(intersection)/len(union)
print(jaccard)

0.007751937984496124


In [61]:
print(intersection)

#only one direct match

{'Apoptosis'}


In [62]:
#function for cross-database name direct matching.

def jaccard_index(db1, db2):
  db1_set = set(db1["name"])
  db2_set = set(db2["name"])

  union = db1_set.union(db2_set)
  intersection = db1_set.intersection(db2_set)

  if len(union) == 0:
    jaccard = 0
  else:
    jaccard = len(intersection)/len(union)

  return jaccard

In [82]:
databases = [
    ("GO:BP", df_gobp),
    ("GO:CC", df_gocc),
    ("GO:MF", df_gomf),
    ("KEGG", df_kegg),
    ("REAC", df_reac)
]

db_names = [name for name, df in databases]
matrix_df = pd.DataFrame(index=db_names, columns=db_names, dtype=float)

for i in range(len(databases)):
    name_i, df_i = databases[i]
    for j in range(i+1, len(databases)):
        name_j, df_j = databases[j]
        score = jaccard_index(df_i, df_j)
        matrix_df.at[name_i, name_j] = score
        matrix_df.at[name_j, name_i] = score

print(matrix_df)

       GO:BP  GO:CC  GO:MF      KEGG      REAC
GO:BP    NaN    0.0    0.0  0.000000  0.000000
GO:CC    0.0    NaN    0.0  0.000000  0.000000
GO:MF    0.0    0.0    NaN  0.000000  0.000000
KEGG     0.0    0.0    0.0       NaN  0.007752
REAC     0.0    0.0    0.0  0.007752       NaN


**Results indicate that direct name comparison yeild nearly zero consistency**